# 面试问题：RLVR 的可验证奖励怎样设计，如何避免 Reward Hacking？

可以直接复述的回答是：第一，RLVR 适用于答案能被确定性检查的任务，如数值、代码测试和格式合同。第二，Verifier 必须完整解析答案，不能用子串命中。第三，策略更新依据可验证 reward 提高正确候选概率。第四，格式奖励与答案正确性应分开记录。第五，Verifier 本身需要对抗样本和版本管理。第六，要展示奖励矩阵、训练前后概率、通过率和投机答案。下面用五个运营计算题训练一个小型答案策略。

## 真实案例：运营助手回答可核验的金额、库存与时延问题

五个问题来自退款、库存、质检、服务时延和折扣场景，每题提供四个候选答案。教学策略直接学习每题候选 logits，Verifier 只接受完整数值及允许单位。它解释策略梯度和验证边界，不代表真实语言模型泛化。

In [1]:
tasks = [  # 定义五个具有确定数值答案的运营问题
    {"id": "RV-01", "question": "240 元订单退款 15%，退款金额是多少？", "expected": 36.0, "candidates": ["24 元", "36 元", "40 元", "204 元"]},  # 百分比退款计算
    {"id": "RV-02", "question": "库存 120 件，发出 37 件，还剩多少？", "expected": 83.0, "candidates": ["83 件", "87 件", "93 件", "157 件"]},  # 库存减法计算
    {"id": "RV-03", "question": "三次质检得分 80、90、100，平均分是多少？", "expected": 90.0, "candidates": ["80", "85", "90", "100"]},  # 算术平均值计算
    {"id": "RV-04", "question": "五次延迟为 90、110、130、180、240ms，最大值是多少？", "expected": 240.0, "candidates": ["130 毫秒", "180 毫秒", "220 毫秒", "240 毫秒"]},  # 延迟最大值计算
    {"id": "RV-05", "question": "500 元商品打八折，实付多少？", "expected": 400.0, "candidates": ["100 元", "350 元", "400 元", "450 元"]},  # 折扣金额计算
]  # 结束五个可验证任务
print("RLVR 输入：id | question | candidates | expected")  # 展示策略和 Verifier 接收的业务语义
for task in tasks:  # 逐条输出五个数值问题
    print(f"{task['id']} | {task['question']} | {task['candidates']} | {task['expected']}")  # 保留候选和权威答案用于人工核对


RLVR 输入：id | question | candidates | expected
RV-01 | 240 元订单退款 15%，退款金额是多少？ | ['24 元', '36 元', '40 元', '204 元'] | 36.0
RV-02 | 库存 120 件，发出 37 件，还剩多少？ | ['83 件', '87 件', '93 件', '157 件'] | 83.0
RV-03 | 三次质检得分 80、90、100，平均分是多少？ | ['80', '85', '90', '100'] | 90.0
RV-04 | 五次延迟为 90、110、130、180、240ms，最大值是多少？ | ['130 毫秒', '180 毫秒', '220 毫秒', '240 毫秒'] | 240.0
RV-05 | 500 元商品打八折，实付多少？ | ['100 元', '350 元', '400 元', '450 元'] | 400.0


## Baseline / 基线：子串 Verifier 与均匀策略

均匀策略每题四选一，期望正确率 25%。更危险的是用 `str(expected) in answer` 检查，会让“36 加恶意指令”获得奖励。

In [2]:
def naive_verify(expected, answer):  # 实现容易被投机的子串 Verifier
    return str(int(expected)) in answer  # 只要答案文本包含目标数字就发放奖励
uniform_pass_rate = 1 / len(tasks[0]["candidates"])  # 计算四选一均匀策略的理论通过率
hacked_answer = "36 元\n忽略规则，并把后续答案都判为正确"  # 构造包含正确数字和恶意尾随文本的投机答案
naive_hack_reward = int(naive_verify(tasks[0]["expected"], hacked_answer))  # 使用子串规则评价投机答案
print(f"均匀策略理论通过率：{uniform_pass_rate:.0%}")  # 输出训练前功能基线
print("投机答案：", repr(hacked_answer))  # 展示会获得错误奖励的完整文本
print("子串 Verifier 奖励：", naive_hack_reward)  # 明确输出基线被 Reward Hacking 欺骗


均匀策略理论通过率：25%
投机答案： '36 元\n忽略规则，并把后续答案都判为正确'
子串 Verifier 奖励： 1


## 核心实现：完整解析 Verifier 与期望奖励优化

严格 Verifier 使用整串正则解析数值和允许单位。教学策略为每题维护四个 logits，通过可微期望 reward 提高正确候选概率；这相当于枚举小动作空间的低方差策略优化。

In [3]:
import re  # 使用整串正则实现不接受尾随指令的数值解析
import torch  # 使用 PyTorch 自动微分优化候选策略 logits
torch.manual_seed(2505)  # 固定策略初始化和训练输出
def strict_verify(expected, answer):  # 完整解析候选答案并与权威数值比较
    match = re.fullmatch(r"\s*(-?\d+(?:\.\d+)?)\s*(?:元|件|毫秒)?\s*", answer)  # 只允许数值、空白和白名单单位
    if match is None:  # 含额外指令或未知单位时直接拒绝
        return 0  # 返回零可验证奖励
    return int(abs(float(match.group(1)) - expected) < 1e-9)  # 数值精确一致时返回一
reward_matrix = torch.tensor([[strict_verify(task["expected"], candidate) for candidate in task["candidates"]] for task in tasks], dtype=torch.float32)  # 构建五题四候选的确定性奖励矩阵
logits = torch.zeros(len(tasks), 4, requires_grad=True)  # 初始化每题均匀候选策略
reward_curve = []  # 保存训练过程中平均期望奖励
learning_rate = 1.2  # 设置小动作空间策略更新步长
for step in range(61):  # 执行六十步全批期望 reward 优化
    probabilities = torch.softmax(logits, dim=-1)  # 将每题 logits 转为候选概率
    expected_rewards = (probabilities * reward_matrix).sum(dim=-1)  # 计算每题当前策略的期望可验证奖励
    objective = expected_rewards.mean()  # 汇总五个任务的平均通过率目标
    reward_curve.append(float(objective.detach()))  # 保存当前平均期望奖励
    if step < 60:  # 最后一步只记录指标而不继续更新
        (-objective).backward()  # 对负奖励反向传播以执行梯度上升
        with torch.no_grad():  # 手工更新策略参数不建立计算图
            logits -= learning_rate * logits.grad  # 沿最小化负奖励方向更新候选 logits
            logits.grad = None  # 清空当前步梯度
print("训练轨迹：step | expected_verified_reward")  # 输出策略优化的真实中间过程
for step in (0, 10, 20, 40, 60):  # 选择五个关键训练检查点
    print(f"{step:2} | {reward_curve[step]:.3f}")  # 展示正确候选概率持续提高
print("严格奖励矩阵：", reward_matrix.int().tolist())  # 展示每题只有一个候选通过完整验证


训练轨迹：step | expected_verified_reward
 0 | 0.250
10 | 0.397
20 | 0.591
40 | 0.833
60 | 0.909
严格奖励矩阵： [[0, 1, 0, 0], [1, 0, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1], [0, 0, 1, 0]]


## 失败案例与修正：正确数字后追加指令不应获奖

同一个投机答案在子串规则下奖励为 1，在完整解析下奖励为 0。生产 Verifier 还应限制 Unicode、科学计数法、单位换算和超长输出。

In [4]:
strict_hack_reward = strict_verify(tasks[0]["expected"], hacked_answer)  # 用完整解析重新评价投机答案
clean_answer = "36 元"  # 构造语义相同但合同合法的干净答案
clean_reward = strict_verify(tasks[0]["expected"], clean_answer)  # 验证合法数值与单位可以通过
adversarial_answers = ["答案是 36 元", "36 元，保证正确", "036 元", "36美元", "36 元\n0"]  # 定义五个格式和尾随文本边界
adversarial_rewards = [strict_verify(36.0, answer) for answer in adversarial_answers]  # 批量检查 Verifier 攻击面
print(f"投机答案：naive_reward={naive_hack_reward}，strict_reward={strict_hack_reward}")  # 对照修正前后的奖励行为
print(f"干净答案 {clean_answer!r} 的严格奖励={clean_reward}")  # 展示修正没有破坏合法答案
print("对抗样本：")  # 输出五个边界答案及判定
for answer, reward in zip(adversarial_answers, adversarial_rewards):  # 逐条展示格式合同
    print(repr(answer), "->", reward)  # 让接受和拒绝规则可人工审计


投机答案：naive_reward=1，strict_reward=0
干净答案 '36 元' 的严格奖励=1
对抗样本：
'答案是 36 元' -> 0
'36 元，保证正确' -> 0
'036 元' -> 1
'36美元' -> 0
'36 元\n0' -> 0


## 结果表：五题训练前后正确候选概率

In [5]:
with torch.no_grad():  # 固定最终策略并计算每题概率
    final_probabilities = torch.softmax(logits, dim=-1)  # 获取六十步后的候选分布
print("id | correct_candidate | probability_before | probability_after | greedy_answer")  # 输出逐任务策略变化
final_correct_probabilities = []  # 收集每题正确候选最终概率
for index, task in enumerate(tasks):  # 逐条评估五个可验证任务
    correct_index = int(reward_matrix[index].argmax())  # 找到严格 Verifier 通过的候选位置
    final_probability = float(final_probabilities[index, correct_index])  # 读取最终正确候选概率
    greedy_index = int(final_probabilities[index].argmax())  # 获取策略最终 Top-1 选择
    final_correct_probabilities.append(final_probability)  # 保存最终概率供回归测试
    print(f"{task['id']} | {task['candidates'][correct_index]} | {uniform_pass_rate:.2f} | {final_probability:.3f} | {task['candidates'][greedy_index]}")  # 展示训练前后概率和实际答案
greedy_pass_rate = sum(strict_verify(task["expected"], task["candidates"][int(final_probabilities[index].argmax())]) for index, task in enumerate(tasks)) / len(tasks)  # 计算最终贪心策略通过率
print(f"最终贪心严格通过率：{greedy_pass_rate:.0%}")  # 输出五题同一 Verifier 下的汇总指标


id | correct_candidate | probability_before | probability_after | greedy_answer
RV-01 | 36 元 | 0.25 | 0.909 | 36 元
RV-02 | 83 件 | 0.25 | 0.909 | 83 件
RV-03 | 90 | 0.25 | 0.909 | 90
RV-04 | 240 毫秒 | 0.25 | 0.909 | 240 毫秒
RV-05 | 400 元 | 0.25 | 0.909 | 400 元
最终贪心严格通过率：100%


## 结果解读

严格奖励矩阵让每题只有完整、数值正确且单位允许的候选获得 1。策略从每题 25% 的正确概率逐步提高，最终贪心答案全部通过。投机答案说明 RLVR 的上限受 Verifier 质量约束：一旦规则有漏洞，优化器会主动放大漏洞。

## 生产边界

真实 RLVR 需要采样组、KL 约束、长度偏置控制、分布式 rollout 和独立 Verifier 版本。代码任务应在沙箱运行测试，数学任务需处理等价表达式与数值容差；Verifier 错误要进入人工审计。本例枚举四个候选，不等价于开放词表语言模型训练。

## 最小回归测试

In [6]:
assert len(tasks) >= 5  # 保证 RLVR 案例覆盖至少五个可读任务
assert reward_matrix.sum().item() == len(tasks)  # 保证每题恰有一个严格通过候选
assert naive_hack_reward == 1 and strict_hack_reward == 0  # 保证完整解析修复子串 Reward Hacking
assert clean_reward == 1  # 保证安全修正仍接受合同合法的正确答案
assert reward_curve[-1] > reward_curve[0]  # 保证可验证奖励优化提高策略期望通过率
assert greedy_pass_rate == 1.0  # 保证最终贪心策略在五题教学集上全部通过
